## Import Libraries and Packages

In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


from typing import Dict, List

## Import the Dataset

In [2]:
ds = load_dataset("AbstractTTS/IEMOCAP")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/489M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/456M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/462M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10039 [00:00<?, ? examples/s]

In [ ]:
# Remove audio column
ds = ds.remove_columns(["audio"])

print(ds["train"].column_names)

['file', 'frustrated', 'angry', 'sad', 'disgust', 'excited', 'fear', 'neutral', 'surprise', 'happy', 'EmoAct', 'EmoVal', 'EmoDom', 'gender', 'transcription', 'major_emotion', 'speaking_rate', 'pitch_mean', 'pitch_std', 'rms', 'relative_db']


In [4]:
cols = ['transcription', 'major_emotion']

df = ds["train"].to_pandas()
df = df[cols]

In [5]:
df.head()

,transcription,major_emotion
0,Excuse me.,neutral
1,Yeah.,neutral
2,Is there a problem?,neutral
3,You did.,neutral
4,You were standing at the beginning and you di...,neutral


### Examine Missing Value

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10039 entries, 0 to 10038
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   transcription  10039 non-null  object
 1   major_emotion  10039 non-null  object
dtypes: object(2)
memory usage: 157.0+ KB


In [7]:
df.describe().T

,count,unique,top,freq
transcription,10039,8068,Yeah.,84
major_emotion,10039,10,frustrated,2917


In [ ]:
print(f"\nTotal samples: {len(df)}")
print(f"\n{df.isnull().sum()}")

# Distribution of emotions
if 'major_emotion' in df.columns:
    print("\nEmotion Distribution:")
    print(df['major_emotion'].value_counts())


Total samples: 10039

transcription    0
major_emotion    0
dtype: int64

Emotion Distribution:
major_emotion
frustrated    2917
excited       1976
neutral       1726
angry         1269
sad           1250
happy          656
surprise       110
fear           107
other           26
disgust          2
Name: count, dtype: int64


In [9]:
# Checking for Empty strings
print(f"transcription : {len(df[df["transcription"] == " "])}")
print(f"major_emotion : {len(df[df["major_emotion"] == " "])}")

transcription : 0
major_emotion : 0


### Sample Transcriptions

In [ ]:
for i, text in enumerate(df['transcription'].head(15), 1):
    print(f"{i:2d}. {text}")

 1.  Excuse me.
 2.  Yeah.
 3.  Is there a problem?
 4.  You did.
 5.  You were standing at the beginning and you directed me.
 6.  Well what's the problem?  Let me change it.
 7.  What?  I'm getting an ID.  This is why I'm here.  My wallet was stolen.
 8.  How am I supposed to get an ID without an ID?  How does a person get an ID in the first place?
 9.  I'm here to get an ID.
10.  Like what?  Like a birth certificate?
11.  Who the hell has a birth certificate?
12.  Yes but my wallet was stolen, I don't have anything.  I don't have any credit cards, I don't have my ID.  Don't you have things on file here?
13.  That's out of control.
14.  How long have you been working here?
15.  Clearly.  You know, do you have like a supervisor or something?


## Build the Confident Feature Extractor

In [11]:
class ConfidenceFeatureExtractor:
    """Enhanced version with comprehensive linguistic markers"""
    
    def __init__(self):
        # HEDGING WORDS - 60+ markers
        self.hedging_words = [
            # Basic hedges
            'maybe', 'perhaps', 'probably', 'possibly', 'potentially',
            'might', 'may', 'could', 'would', 'should',
            # Cognitive hedges
            'i think', 'i believe', 'i feel', 'i guess', 'i suppose',
            'i imagine', 'i assume', 'i reckon', 'i suspect',
            # Degree hedges
            'kind of', 'sort of', 'somewhat', 'rather', 'fairly',
            'quite', 'relatively', 'comparatively', 'reasonably',
            'pretty much', 'more or less', 'to some extent',
            # Approximators
            'approximately', 'roughly', 'about', 'around', 'nearly',
            'almost', 'practically', 'virtually', 'essentially',
            # Probability hedges
            'likely', 'unlikely', 'apt to', 'liable to', 'tends to',
            'seems to', 'appears to', 'looks like', 'sounds like',
            # Conditional hedges
            'if possible', 'if available', 'if applicable', 'depending on',
            'subject to', 'provided that', 'assuming that',
            # Minimizers
            'a bit', 'a little', 'slightly', 'marginally',
            'to a degree', 'in a way', 'in a sense',
            # Tentative language
            'tend to', 'seem to', 'appear to', 'suggest that',
            'indicate that', 'imply that',
            # Epistemic hedges
            'as far as i know', 'to my knowledge', 'in my opinion',
            'in my view', 'from my perspective', 'it seems that',
            'it appears that', 'i would say',
        ]
        
        # FILLER WORDS - 50+ markers
        self.filler_words = [
            # Classic fillers
            'um', 'uh', 'er', 'ah', 'eh', 'mm', 'hmm', 'uhh', 'umm',
            # Discourse markers
            'like', 'you know', 'i mean', 'you see',
            'you know what i mean', 'know what i\'m saying',
            # Temporal fillers
            'well', 'so', 'now', 'then', 'anyway', 'anyways',
            'alright', 'okay', 'ok',
            # Intensifiers used as fillers
            'actually', 'basically', 'literally', 'seriously',
            'honestly', 'really', 'totally', 'completely',
            # Repair markers
            'i mean to say', 'what i mean is', 'let me think',
            'how do i put it', 'how should i say',
            # Stalling phrases
            'let me see', 'let me check', 'give me a second',
            'just a moment', 'hold on', 'wait a minute',
            'one second', 'bear with me',
            # Continuation markers
            'and everything', 'and stuff', 'and all that',
            'or something', 'or whatever', 'and so on',
            'et cetera', 'and the like',
        ]
        
        # CONFIDENCE WORDS - 70+ markers
        self.confidence_words = [
            # Certainty markers
            'definitely', 'certainly', 'absolutely', 'undoubtedly',
            'without doubt', 'no doubt', 'for sure', 'for certain',
            # Knowledge indicators
            'know', 'knows', 'knew', 'knowing', 'understand',
            'clear', 'clearly', 'obvious', 'obviously', 'evident',
            'evidently', 'apparent', 'apparently',
            # Assurance words
            'sure', 'positive', 'confident', 'certain', 'convinced',
            'assured', 'secure', 'firm',
            # Future commitment
            'will', 'shall', 'going to', 'guarantee', 'guaranteed',
            'promise', 'promised',
            # Strong affirmatives
            'yes', 'correct', 'right', 'exactly', 'precisely',
            'indeed', 'truly', 'surely',
            # Expertise markers
            'expert', 'experienced', 'qualified', 'trained',
            'specialized', 'professional', 'competent',
            # Factual language
            'fact', 'facts', 'proven', 'verified', 'confirmed',
            'established', 'demonstrated', 'shown',
            # Direct statements
            'is', 'are', 'was', 'were', 'been',
            # Solution-oriented
            'solution', 'solve', 'fix', 'resolve', 'handle',
            'address', 'deal with', 'take care of',
            # Immediacy
            'now', 'immediately', 'right away', 'right now',
            'instantly', 'straightaway', 'at once',
        ]
        
        # UNCERTAINTY PHRASES - 30+ markers
        self.uncertainty_phrases = [
            # Direct uncertainty
            "i don't know", "i dunno", "i'm not sure", "not sure",
            "i'm uncertain", "i'm unsure", "i have no idea",
            # Confusion
            "i'm confused", "that's confusing", "unclear",
            "i don't understand", "i'm lost",
            # Lack of information
            "i don't have", "i lack", "i'm missing",
            "can't tell", "hard to say", "difficult to say",
            # Inability
            "i can't", "i cannot", "unable to", "not able to",
            "don't know how",
            # Doubt
            "i doubt", "doubtful", "questionable",
            # Need for help
            "need help", "need assistance", "need to check",
            "need to ask", "need to verify",
            # Deferral
            "ask someone else", "check with", "another department",
        ]
        
        # ASSERTIVE WORDS - 25+ markers
        self.assertive_words = [
            'must', 'have to', 'need to', 'got to', 'gotta',
            'ought to', 'should', 'shall', 'will',
            'require', 'required', 'requires', 'necessary',
            'essential', 'crucial', 'critical', 'vital',
            'important', 'imperative', 'mandatory',
            'ensure', 'make sure', 'guarantee',
        ]
        
        # POLITENESS MARKERS - 20+ markers
        self.politeness_markers = [
            'sorry', 'apologize', 'apologies', 'excuse me',
            'pardon', 'please', 'kindly', 'thank you',
            'thanks', 'appreciate', 'grateful',
            'would you mind', 'could you please', 'if possible',
            'with respect', 'respectfully', 'if i may',
        ]
        
        # ABSOLUTE WORDS - 30+ markers
        self.absolute_words = [
            'all', 'every', 'everyone', 'everybody', 'everything',
            'always', 'forever', 'constantly', 'continually',
            'none', 'nothing', 'nobody', 'no one', 'nowhere',
            'never', 'complete', 'completely', 'total', 'totally',
            'entire', 'entirely', 'whole', 'wholly',
            'full', 'fully', 'absolute', 'absolutely',
        ]
        
        # QUESTION WORDS - 15+ markers
        self.question_words = [
            'what', 'when', 'where', 'why', 'who', 'whom',
            'which', 'whose', 'how',
            'is it', 'are there', 'do you', 'can you',
            'could you', 'would you',
        ]
        
        # POSITIVE EMOTIONS - 13 markers
        self.positive_emotion_words = [
            'happy', 'glad', 'pleased', 'delighted', 'excited',
            'confident', 'proud', 'satisfied', 'content',
            'great', 'excellent', 'perfect', 'wonderful',
        ]
        
        # NEGATIVE EMOTIONS - 12 markers
        self.negative_emotion_words = [
            'worried', 'concerned', 'anxious', 'nervous',
            'hesitant', 'reluctant', 'uncomfortable', 'uneasy',
            'stressed', 'frustrated', 'confused', 'unsure',
        ]
        
        # ESCALATION PHRASES - 9 markers
        self.escalation_phrases = [
            'transfer you', 'transfer to', 'supervisor',
            'manager', 'specialist', 'someone who can',
            'better suited', 'another department', 'escalate',
        ]
        
        # OWNERSHIP PHRASES - 10 markers
        self.ownership_phrases = [
            'i will', 'i can', "i'll handle", "i'll take care",
            'let me help', 'i can help', "i'll assist",
            "i'll resolve", "i'll fix", "i'll solve",
        ]
        
        # DELAY PHRASES - 9 markers
        self.delay_phrases = [
            'call back', 'get back to you', 'follow up',
            'check on that', 'look into', 'investigate',
            'find out', 'research', 'verify',
        ]
    
    def extract_features(self, text: str) -> Dict[str, float]:
        """Extract comprehensive confidence features"""
        text_lower = str(text).lower()
        words = text_lower.split()
        word_count = max(len(words), 1)
        
        features = {}
        
        # Basic ratios
        features['hedging_ratio'] = self._count_matches(text_lower, self.hedging_words) / word_count
        features['filler_ratio'] = self._count_matches(text_lower, self.filler_words) / word_count
        features['confidence_ratio'] = self._count_matches(text_lower, self.confidence_words) / word_count
        features['assertive_ratio'] = self._count_matches(text_lower, self.assertive_words) / word_count
        features['politeness_ratio'] = self._count_matches(text_lower, self.politeness_markers) / word_count
        features['absolute_ratio'] = self._count_matches(text_lower, self.absolute_words) / word_count
        
        # Phrase counts
        features['uncertainty_count'] = self._count_matches(text_lower, self.uncertainty_phrases)
        features['question_word_count'] = self._count_matches(text_lower, self.question_words)
        
        # Emotional markers
        features['positive_emotion_ratio'] = self._count_matches(text_lower, self.positive_emotion_words) / word_count
        features['negative_emotion_ratio'] = self._count_matches(text_lower, self.negative_emotion_words) / word_count
        
        # Call center specific
        features['escalation_count'] = self._count_matches(text_lower, self.escalation_phrases)
        features['ownership_count'] = self._count_matches(text_lower, self.ownership_phrases)
        features['delay_count'] = self._count_matches(text_lower, self.delay_phrases)
        
        # Punctuation
        features['question_ratio'] = text.count('?') / word_count
        features['exclamation_ratio'] = text.count('!') / word_count
        
        # Sentence structure
        features['avg_word_length'] = np.mean([len(w) for w in words]) if words else 0
        features['sentence_length'] = word_count
        
        # First person
        first_person = ['i ', 'me ', 'my ', 'mine ', 'myself ']
        features['first_person_ratio'] = sum(1 for fp in first_person if fp in text_lower + ' ') / word_count
        
        # Statement type
        features['is_statement'] = 1 if not text.strip().endswith('?') else 0
        
        # Composite score
        features['confidence_score'] = (
            features['confidence_ratio'] + 
            features['assertive_ratio'] +
            features['ownership_count'] / word_count -
            features['hedging_ratio'] -
            features['filler_ratio'] -
            features['uncertainty_count'] / word_count
        )
        
        return features
    
    def _count_matches(self, text: str, word_list: List[str]) -> int:
        """Count matches in text"""
        return sum(1 for item in word_list if item in text)
    
    def extract_batch(self, texts: List[str]) -> pd.DataFrame:
        """Extract features for multiple texts"""
        features_list = [self.extract_features(text) for text in texts]
        return pd.DataFrame(features_list)
    
    def get_feature_summary(self) -> pd.DataFrame:
        """Get summary of all markers"""
        summary_data = [
            {'Category': 'Hedging Words', 'Count': len(self.hedging_words)},
            {'Category': 'Filler Words', 'Count': len(self.filler_words)},
            {'Category': 'Confidence Words', 'Count': len(self.confidence_words)},
            {'Category': 'Uncertainty Phrases', 'Count': len(self.uncertainty_phrases)},
            {'Category': 'Assertive Words', 'Count': len(self.assertive_words)},
            {'Category': 'Politeness Markers', 'Count': len(self.politeness_markers)},
            {'Category': 'Absolute Words', 'Count': len(self.absolute_words)},
            {'Category': 'Question Words', 'Count': len(self.question_words)},
            {'Category': 'Positive Emotions', 'Count': len(self.positive_emotion_words)},
            {'Category': 'Negative Emotions', 'Count': len(self.negative_emotion_words)},
            {'Category': 'Escalation Phrases', 'Count': len(self.escalation_phrases)},
            {'Category': 'Ownership Phrases', 'Count': len(self.ownership_phrases)},
            {'Category': 'Delay Phrases', 'Count': len(self.delay_phrases)},
        ]
        return pd.DataFrame(summary_data)


### Test Feature Extraction

In [ ]:
extractor = ConfidenceFeatureExtractor()

test_sentences = [
    "I definitely know the answer and will help you right away.",
    "Um, I think maybe we could possibly try to help with that.",
    "I don't know, perhaps someone else might be able to assist you."
]

for i, sentence in enumerate(test_sentences, 1):
    print(f"\nExample {i}: \"{sentence}\"")
    features = extractor.extract_features(sentence)
    print(f"  Confidence words: {features['confidence_ratio']:.3f}")
    print(f"  Hedging words: {features['hedging_ratio']:.3f}")
    print(f"  Filler words: {features['filler_ratio']:.3f}")
    print(f"  Assertiveness: {features['assertive_ratio']:.3f}")
    print(f"  Question?: {'Yes' if features['question_ratio'] > 0 else 'No'}")


Example 1: "I definitely know the answer and will help you right away."
  Confidence words: 0.545
  Hedging words: 0.000
  Filler words: 0.182
  Assertiveness: 0.091
  Question?: No

Example 2: "Um, I think maybe we could possibly try to help with that."
  Confidence words: 0.000
  Hedging words: 0.417
  Filler words: 0.083
  Assertiveness: 0.000
  Question?: No

Example 3: "I don't know, perhaps someone else might be able to assist you."
  Confidence words: 0.250
  Hedging words: 0.167
  Filler words: 0.250
  Assertiveness: 0.000
  Question?: No


## Build the Confident Detector

In [13]:
class ConfidenceDetector:
    """Main confidence detection model using unsupervised learning"""
    
    def __init__(self, n_clusters=3):
        self.n_clusters = n_clusters
        self.feature_extractor = ConfidenceFeatureExtractor()
        self.scaler = StandardScaler()
        self.kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        self.tfidf = TfidfVectorizer(max_features=100, ngram_range=(1, 2))
        self.cluster_labels = {}  # Maps cluster IDs to confidence levels
        self.feature_names = None
        
    def preprocess_data(self, df):
        """Clean and prepare the dataset"""
        df = df.copy()
        
        # Clean transcriptions
        df['transcription_clean'] = df['transcription'].apply(self._clean_text)
        
        return df
    
    def _clean_text(self, text):
        """Clean text data"""
        text = str(text).lower()
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    def fit(self, df):
        """Train the confidence detection model"""
        print("\nTraining Confidence Detector...")
        print("=" * 80)
        
        # Preprocess
        df = self.preprocess_data(df)
        print(f"  Dataset shape after preprocessing: {df.shape}")
        
        # Extract linguistic features
        print("  Extracting linguistic features...")
        linguistic_features = self.feature_extractor.extract_batch(df['transcription_clean'].tolist())
        self.feature_names = linguistic_features.columns.tolist()
        
        # Extract TF-IDF features
        print("  Extracting TF-IDF features...")
        tfidf_features = self.tfidf.fit_transform(df['transcription_clean']).toarray()
        
        # Combine features
        print("  Combining features...")
        combined_features = np.hstack([
            linguistic_features.values,
            tfidf_features
        ])
        
        # Scale features
        print("  Scaling features...")
        scaled_features = self.scaler.fit_transform(combined_features)
        
        # Cluster
        print(f"  Clustering into {self.n_clusters} confidence levels...")
        self.kmeans.fit(scaled_features)
        
        # Calculate silhouette score
        silhouette = silhouette_score(scaled_features, self.kmeans.labels_)
        print(f"  Silhouette Score: {silhouette:.3f} (higher is better)")
        
        # Assign cluster labels
        self._assign_confidence_labels(linguistic_features, self.kmeans.labels_)
        
        print("\n✅ Training complete!")
        return self
    
    def _assign_confidence_labels(self, linguistic_features, cluster_labels):
        """Assign cluster IDs to confidence levels based on characteristics"""
        cluster_stats = {}
        
        for cluster_id in range(self.n_clusters):
            mask = cluster_labels == cluster_id
            cluster_data = linguistic_features[mask]
            
            # Calculate confidence score
            confidence_score = (
                cluster_data['confidence_ratio'].mean() +
                cluster_data['assertive_ratio'].mean() +
                cluster_data['is_statement'].mean() -
                cluster_data['hedging_ratio'].mean() -
                cluster_data['filler_ratio'].mean() -
                cluster_data['question_ratio'].mean() -
                cluster_data['uncertainty_count'].mean()
            )
            
            cluster_stats[cluster_id] = confidence_score
        
        # Sort clusters by confidence score
        sorted_clusters = sorted(cluster_stats.items(), key=lambda x: x[1])
        
        # Assign labels
        self.cluster_labels = {
            sorted_clusters[0][0]: 'Low',
            sorted_clusters[1][0]: 'Medium' if self.n_clusters == 3 else 'Low-Medium',
        }
        
        if self.n_clusters == 3:
            self.cluster_labels[sorted_clusters[2][0]] = 'High'
        elif self.n_clusters >= 4:
            for i in range(2, self.n_clusters):
                if i == self.n_clusters - 1:
                    self.cluster_labels[sorted_clusters[i][0]] = 'High'
                else:
                    self.cluster_labels[sorted_clusters[i][0]] = f'Medium-{i-1}'
        
        print("\n  Cluster to Confidence mapping:")
        for cluster_id, label in self.cluster_labels.items():
            print(f"    Cluster {cluster_id} -> {label} (score: {cluster_stats[cluster_id]:.3f})")
    
    def predict(self, texts):
        """Predict confidence levels for new texts"""
        # Extract features
        linguistic_features = self.feature_extractor.extract_batch(texts)
        tfidf_features = self.tfidf.transform(texts).toarray()
        
        # Combine and scale
        combined_features = np.hstack([
            linguistic_features.values,
            tfidf_features
        ])
        scaled_features = self.scaler.transform(combined_features)
        
        # Predict clusters
        cluster_predictions = self.kmeans.predict(scaled_features)
        
        # Map to confidence labels
        confidence_predictions = [self.cluster_labels[cluster] for cluster in cluster_predictions]
        
        return confidence_predictions
    
    def predict_with_scores(self, texts):
        """Predict with detailed feature scores"""
        predictions = self.predict(texts)
        linguistic_features = self.feature_extractor.extract_batch(texts)
        
        results = pd.DataFrame({
            'transcription': texts,
            'confidence_level': predictions,
            'hedging_ratio': linguistic_features['hedging_ratio'],
            'filler_ratio': linguistic_features['filler_ratio'],
            'confidence_words': linguistic_features['confidence_ratio'],
            'assertiveness': linguistic_features['assertive_ratio']
        })
        
        return results

## Train the Model

In [14]:
detector = ConfidenceDetector(n_clusters=3)
detector.fit(df)


Training Confidence Detector...
  Dataset shape after preprocessing: (10039, 3)
  Extracting linguistic features...
  Extracting TF-IDF features...
  Combining features...
  Scaling features...
  Clustering into 3 confidence levels...
  Silhouette Score: 0.048 (higher is better)

  Cluster to Confidence mapping:
    Cluster 1 -> Low (score: 0.448)
    Cluster 0 -> Medium (score: 0.604)
    Cluster 2 -> High (score: 0.655)

✅ Training complete!


## Generate Predictions

In [15]:
results = detector.predict_with_scores(df['transcription'].tolist())
results.head(20)

,transcription,confidence_level,hedging_ratio,filler_ratio,confidence_words,assertiveness
0,Excuse me.,High,0.000000,0.000000,0.000000,0.0
1,Yeah.,High,0.000000,1.000000,0.000000,0.0
2,Is there a problem?,High,0.000000,0.250000,0.250000,0.0
3,You did.,High,0.000000,0.000000,0.000000,0.0
4,You were standing at the beginning and you di...,High,0.000000,0.100000,0.100000,0.0
5,Well what's the problem? Let me change it.,High,0.000000,0.125000,0.000000,0.0
6,What? I'm getting an ID. This is why I'm he...,High,0.000000,0.071429,0.142857,0.0
7,How am I supposed to get an ID without an ID?...,Low,0.045455,0.090909,0.000000,0.0
8,I'm here to get an ID.,High,0.000000,0.166667,0.000000,0.0
9,Like what? Like a birth certificate?,High,0.000000,0.333333,0.000000,0.0


### Confident Level Distribuition

In [ ]:
print("Confidence Level Distribution:")

conf_dist = results['confidence_level'].value_counts()
print("\nCounts:")
print(conf_dist)

print("\nPercentages:")
print((conf_dist / len(results) * 100).round(2))

Confidence Level Distribution:

Counts:
confidence_level
High      6777
Low       2705
Medium     557
Name: count, dtype: int64

Percentages:
confidence_level
High      67.51
Low       26.94
Medium     5.55
Name: count, dtype: float64


### Example Transcriptions by Confidence Level

In [17]:
for level in ['Low', 'Medium', 'High']:
    mask = results['confidence_level'] == level
    if mask.sum() > 0:
        examples = results[mask]['transcription'].head(10)
        print(f"\n{level} Confidence ({mask.sum()} total samples):")
        for i, ex in enumerate(examples, 1):
            print(f"  {i:2d}. {ex}")


Low Confidence (2705 total samples):
   1.  How am I supposed to get an ID without an ID?  How does a person get an ID in the first place?
   2.  Yes but my wallet was stolen, I don't have anything.  I don't have any credit cards, I don't have my ID.  Don't you have things on file here?
   3.  Clearly.  You know, do you have like a supervisor or something?
   4.  Do you have your forms?
   5.  Who told you to get in this line?
   6.  I don't know.  But I need an ID to pass this form along.  I can't just send it along without an ID.
   7.  I don't understand why this is so complicated for people when they get here.  It's just a simple form.  I just need an ID.
   8.  Yeah.  Do you want to see my supervisor?  Huh? Yeah.  Do you want to see my supervisor?  Fine.  I'll be right back.
   9.  I don't know.  I put in that. request too. They didn't...
  10.  I guess, you know, everybody has to make sacrifices.

Medium Confidence (557 total samples):
   1.  You can't--  This is not the line fo

## Export the dataset

In [18]:
results.to_csv('new_confident_dataset.csv', index=False)